In [ ]:
from transformers import pipeline
unmasker = pipeline('fill-mask', model='distilbert-base-uncased')
unmasker("Hello I'm a [MASK] model.")

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

model = AutoModelForTokenClassification.from_pretrained("./skillgraph-ner/checkpoint-3000")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
import json
import pandas as pd
from collections import Counter

# Load the extractions
with open("raw_extractions.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Total extractions: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

In [ ]:
# Breakdown by type (Skill vs Knowledge)
print("=== Entity Type Distribution ===")
print(df["type"].value_counts())
print(f"\nSkill/Knowledge ratio: {(df['type'] == 'Skill').sum() / (df['type'] == 'Knowledge').sum():.2f}")

In [ ]:
# Top 25 most common skills/knowledge
print("=== Top 25 Most Common Entities ===")
df["text_clean"] = df["text"].str.lower().str.strip()
top_entities = df["text_clean"].value_counts().head(25)
print(top_entities)

In [ ]:
# Top skills vs top knowledge separately
print("=== Top 15 Skills ===")
top_skills = df[df["type"] == "Skill"]["text_clean"].value_counts().head(15)
print(top_skills)

print("\n=== Top 15 Knowledge ===")
top_knowledge = df[df["type"] == "Knowledge"]["text_clean"].value_counts().head(15)
print(top_knowledge)

In [ ]:
# Confidence score distribution
print("=== Confidence Score Statistics ===")
print(df["score"].describe())

print("\n=== Confidence by Type ===")
print(df.groupby("type")["score"].describe())

In [ ]:
# High confidence extractions (>0.95) - most reliable
print("=== High Confidence Entities (score > 0.95) ===")
high_conf = df[df["score"] > 0.95]["text_clean"].value_counts().head(20)
print(high_conf)
print(f"\n{len(df[df['score'] > 0.95]):,} extractions with >95% confidence ({len(df[df['score'] > 0.95])/len(df)*100:.1f}%)")

In [ ]:
# Unique entities count
unique_entities = df["text_clean"].nunique()
print(f"=== Uniqueness Stats ===")
print(f"Total extractions: {len(df):,}")
print(f"Unique entities: {unique_entities:,}")
print(f"Average mentions per entity: {len(df)/unique_entities:.1f}")

# Entities mentioned only once (potential noise or rare skills)
single_mentions = df["text_clean"].value_counts()
single_count = (single_mentions == 1).sum()
print(f"\nEntities mentioned only once: {single_count:,} ({single_count/unique_entities*100:.1f}% of unique entities)")

In [ ]:
# Visualizations
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Type distribution pie chart
ax1 = axes[0, 0]
df["type"].value_counts().plot.pie(ax=ax1, autopct='%1.1f%%', colors=['#4CAF50', '#2196F3'])
ax1.set_title("Skill vs Knowledge Distribution")
ax1.set_ylabel("")

# 2. Top 15 entities bar chart
ax2 = axes[0, 1]
df["text_clean"].value_counts().head(15).plot.barh(ax=ax2, color='#673AB7')
ax2.set_title("Top 15 Most Common Entities")
ax2.set_xlabel("Count")
ax2.invert_yaxis()

# 3. Confidence score histogram
ax3 = axes[1, 0]
df["score"].hist(ax=ax3, bins=50, color='#FF9800', edgecolor='white')
ax3.set_title("Confidence Score Distribution")
ax3.set_xlabel("Score")
ax3.set_ylabel("Frequency")

# 4. Confidence by type boxplot
ax4 = axes[1, 1]
df.boxplot(column="score", by="type", ax=ax4)
ax4.set_title("Confidence Score by Entity Type")
plt.suptitle("")

plt.tight_layout()
plt.show()

In [ ]:
# Entity length analysis - are longer phrases better quality?
df["word_count"] = df["text_clean"].str.split().str.len()
print("=== Entity Length Analysis ===")
print(df.groupby("word_count").agg({
    "score": ["mean", "count"],
    "text_clean": lambda x: x.mode().iloc[0] if len(x) > 0 else ""
}).head(10))

print("\n=== Sample Multi-Word Entities ===")
multi_word = df[df["word_count"] >= 2].sort_values("score", ascending=False)
print(multi_word[["text", "type", "score"]].drop_duplicates("text").head(15))